# LLM-Based Answer Generation: Retrieval vs Generation EM/F1

## Why Generation-Based Evaluation Matters

Right now our EM/F1 just checks if the gold answer appears somewhere in the 5 retrieved pages. A real RAG system uses an LLM to READ those pages and ANSWER the question. This notebook adds that missing LLM generation step.

**The dual comparison is the key insight:** High retrieval coverage (the gold answer is somewhere in the retrieved pages) does *not* always translate to correct generated answers. A page may contain the answer string coincidentally, or the LLM may fail to extract the answer even when present.

This notebook evaluates all three retrieval systems (B1 Naive RAG, B2 HtmlRAG-style, HyperRAG) on 10 HotpotQA questions. For each question and system, we compute:

1. **Retrieval-based EM/F1** — does the gold answer string appear in the retrieved context? (existing method)
2. **Generation-based EM/F1** — does the LLM's *generated answer* match the gold answer? (new, the true RAG measure)

In [ ]:
# --- Colab install (uncomment if running on Google Colab) ---
# !pip install transformers torch tqdm beautifulsoup4

import sys
import csv
from pathlib import Path
import torch
from transformers import pipeline as hf_pipeline
from tqdm import tqdm
from bs4 import BeautifulSoup

# Ensure src/ is importable (for Colab or running from notebooks/)
if str(Path("..").resolve()) not in sys.path:
    sys.path.insert(0, str(Path("..").resolve()))

from src.corpus import load_corpus, load_hotpotqa
from src.embeddings import load_index
from src.graph import load_graph
from src.retrieval import naive_rag, htmlrag_style, hyperrag, compute_em, compute_f1

print(f"Python {sys.version}")
print(f"PyTorch {torch.__version__} | CUDA available: {torch.cuda.is_available()}")

In [ ]:
# === CONFIGURATION ===
# Change MODEL_NAME to swap models. TinyLlama is ungated and CPU-friendly.
# For Llama 3.x: requires `huggingface-cli login` + Meta approval.
# MODEL_NAME = "meta-llama/Llama-3.2-3B-Instruct"  # gated — needs HF login
MODEL_NAME = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"  # ungated, ~2.2GB

N_QUESTIONS = 10       # Number of HotpotQA questions to evaluate (indices 0-9)
K = 5                  # Pages retrieved by FAISS
EXPAND_K = 3           # Additional graph-neighbor pages for HyperRAG
MAX_CONTEXT_TOKENS = 1500  # Truncate context to this many tokens (D-08)
MAX_NEW_TOKENS = 20    # Max tokens for generated answer

DATA_DIR = Path("..") / "data"  # Adjust if running from repo root
RESULTS_CSV = DATA_DIR / "llm_eval_results.csv"
FORCE_RERUN = False    # Set True to regenerate even if CSV exists

print(f"Model: {MODEL_NAME}")
print(f"Questions: {N_QUESTIONS} | K={K} | EXPAND_K={EXPAND_K}")
print(f"Max context tokens: {MAX_CONTEXT_TOKENS} | Max new tokens: {MAX_NEW_TOKENS}")
print(f"Results CSV: {RESULTS_CSV}")

## Step 1: Load Data

Load the prebuilt corpus, FAISS index, and hyperlink graph from `data/`. These are created by the earlier phases of HyperRAG-M2.

In [ ]:
print("Loading corpus, FAISS index, and hyperlink graph...")
corpus = load_corpus(DATA_DIR / "corpus.json")
index, _page_ids = load_index(DATA_DIR)
graph = load_graph(DATA_DIR / "hyperlink_graph.graphml")

print(f"Loading {N_QUESTIONS} HotpotQA questions...")
qa_items = load_hotpotqa(split="train", n_samples=N_QUESTIONS)
print(f"Corpus: {len(corpus)} pages | Graph: {graph.number_of_nodes()} nodes, {graph.number_of_edges()} edges")
print(f"Loaded {len(qa_items)} QA pairs")

## Step 2: Load LLM

**Note:** Llama 3.x models are gated — you need to run `huggingface-cli login` and have Meta approval before downloading. TinyLlama is the default (ungated, ~2.2GB). First run downloads the model to `~/.cache/huggingface`.

The pipeline is created **once** and reused for all 30 inference calls (10 questions × 3 systems). Loading a ~1.1B model takes ~30 seconds on CPU; reloading it 30 times would add 15+ minutes to the runtime.

**Runtime note:** On CPU (float32), expect ~5–15 minutes total for 30 inference calls. Use the cached CSV on subsequent runs.

In [ ]:
_bnb_available = False
if torch.cuda.is_available():
    try:
        import bitsandbytes  # noqa: F401
        _bnb_available = True
    except ImportError:
        print("bitsandbytes not available — skipping 4-bit quantization")

if _bnb_available:
    pipe = hf_pipeline(
        "text-generation",
        model=MODEL_NAME,
        device_map="auto",
        model_kwargs={"load_in_4bit": True},
    )
    print(f"Loaded {MODEL_NAME} with 4-bit quantization (GPU)")
else:
    device = 0 if torch.cuda.is_available() else -1
    pipe = hf_pipeline(
        "text-generation",
        model=MODEL_NAME,
        torch_dtype=torch.float32,
        device=device,
    )
    env = "GPU" if torch.cuda.is_available() else "CPU"
    print(f"Loaded {MODEL_NAME} in float32 ({env})")

## Step 3: Helper Functions

Three helper functions prepare context and generate answers:

1. `truncate_to_tokens` — clips context to 1500 tokens using the model's own tokenizer (accurate token counting)
2. `prepare_context_for_llm` — strips HTML from B2 context (LLMs cannot reason efficiently over raw HTML tags)
3. `generate_answer` — constructs the short-answer prompt and calls the pipeline

In [ ]:
def truncate_to_tokens(text: str, tokenizer, max_tokens: int = 1500) -> str:
    """Truncate text to at most max_tokens using the model's tokenizer.

    Truncates from the END — keeps the beginning of the context, which tends
    to contain the most relevant retrieved page (FAISS top-1 is always first).

    Args:
        text: Raw context string from retrieval.
        tokenizer: Model tokenizer (pipe.tokenizer).
        max_tokens: Maximum number of tokens to keep.

    Returns:
        Truncated plain text string.
    """
    tokens = tokenizer.encode(text, add_special_tokens=False)
    if len(tokens) <= max_tokens:
        return text
    return tokenizer.decode(tokens[:max_tokens], skip_special_tokens=True)


def prepare_context_for_llm(context: str, system_name: str) -> str:
    """Strip HTML from B2 (htmlrag_style) context before LLM inference.

    B2 retrieval returns cleaned HTML — suitable for retrieval-based EM/F1
    but harmful for LLM input (the LLM wastes context window on HTML tags).
    Plain text contexts (B1, HyperRAG) are returned unchanged.

    Args:
        context: Retrieved context string.
        system_name: One of 'B1_naive_rag', 'B2_htmlrag', 'HyperRAG'.

    Returns:
        Plain text context string.
    """
    if system_name == "B2_htmlrag" and "<" in context:
        return BeautifulSoup(context, "html.parser").get_text(separator=" ", strip=True)
    return context


def generate_answer(
    question: str,
    context: str,
    pipe,
    max_context_tokens: int = 1500,
    max_new_tokens: int = 20,
) -> str:
    """Generate a short answer from retrieved context using an LLM.

    Truncates context to max_context_tokens, constructs a short-answer prompt,
    and returns the generated answer stripped and lowercased.

    Args:
        question: HotpotQA question string.
        context: Retrieved context (plain text — HTML must be stripped before calling).
        pipe: HuggingFace text-generation pipeline (loaded once, reused).
        max_context_tokens: Maximum tokens to use for context (default 1500).
        max_new_tokens: Maximum tokens to generate (default 20 for short answers).

    Returns:
        Generated answer string, lowercased and stripped.
    """
    truncated_context = truncate_to_tokens(context, pipe.tokenizer, max_context_tokens)

    messages = [
        {
            "role": "system",
            "content": (
                "You are a question-answering assistant. "
                "Answer using ONLY the provided context. "
                "Give a SHORT answer (1-5 words). Do not explain."
            ),
        },
        {
            "role": "user",
            "content": f"Context: {truncated_context}\n\nQuestion: {question}\n\nAnswer:",
        },
    ]

    outputs = pipe(
        messages,
        max_new_tokens=max_new_tokens,
        do_sample=False,
        return_full_text=False,
    )
    return outputs[0]["generated_text"].strip().lower()


print("Helper functions defined: truncate_to_tokens, prepare_context_for_llm, generate_answer")

## Step 4: Run Evaluation

For each of 10 HotpotQA questions, retrieve context from all 3 systems, generate an answer using the LLM, and compute both retrieval-based and generation-based EM/F1.

If `RESULTS_CSV` already exists and `FORCE_RERUN=False`, cached results are loaded instead of re-running inference (saves ~5–15 minutes on repeated runs).

In [ ]:
if RESULTS_CSV.exists() and not FORCE_RERUN:
    print(f"Loading cached results from {RESULTS_CSV}")
    with open(RESULTS_CSV, newline="", encoding="utf-8") as f:
        rows = list(csv.DictReader(f))
    print(f"Loaded {len(rows)} rows")
else:
    systems = {
        "B1_naive_rag": lambda q, k: naive_rag(q, index, corpus, k),
        "B2_htmlrag": lambda q, k: htmlrag_style(q, index, corpus, k),
        "HyperRAG": lambda q, k: hyperrag(q, index, corpus, graph, k=k, expand_k=EXPAND_K),
    }

    rows = []
    total = N_QUESTIONS * len(systems)
    with tqdm(total=total, desc="LLM Evaluation") as pbar:
        for i, qa in enumerate(qa_items):
            question_id = str(qa.get("id", i))
            question = qa["question"]
            gold_answer = qa["answer"]

            for system_name, system_fn in systems.items():
                context = system_fn(question, K)

                # Retrieval-based EM/F1 (existing method)
                retrieval_em = compute_em(context, gold_answer)
                retrieval_f1 = compute_f1(context, gold_answer)

                # Generation-based: prepare context, generate answer, score
                llm_context = prepare_context_for_llm(context, system_name)
                generated_answer = generate_answer(
                    question, llm_context, pipe,
                    MAX_CONTEXT_TOKENS, MAX_NEW_TOKENS
                )
                generation_em = compute_em(generated_answer, gold_answer)
                generation_f1 = compute_f1(generated_answer, gold_answer)

                rows.append({
                    "question_id": question_id,
                    "question": question,
                    "gold_answer": gold_answer,
                    "system": system_name,
                    "retrieval_em": retrieval_em,
                    "retrieval_f1": round(retrieval_f1, 4),
                    "generated_answer": generated_answer,
                    "generation_em": generation_em,
                    "generation_f1": round(generation_f1, 4),
                })
                pbar.update(1)

    # Save to CSV (per D-14)
    DATA_DIR.mkdir(parents=True, exist_ok=True)
    with open(RESULTS_CSV, "w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=[
            "question_id", "question", "gold_answer", "system",
            "retrieval_em", "retrieval_f1", "generated_answer",
            "generation_em", "generation_f1",
        ])
        writer.writeheader()
        writer.writerows(rows)
    print(f"Results saved to {RESULTS_CSV} ({len(rows)} rows)")

## Step 5: Results — Retrieval vs Generation EM/F1

The table below compares per-system average scores for both evaluation methods side by side.

In [ ]:
# Compute per-system averages
system_names = ["B1_naive_rag", "B2_htmlrag", "HyperRAG"]
summary: dict[str, dict[str, float]] = {}

for sname in system_names:
    srows = [r for r in rows if r["system"] == sname]
    if not srows:
        continue
    summary[sname] = {
        "retrieval_em":  sum(float(r["retrieval_em"])  for r in srows) / len(srows),
        "retrieval_f1":  sum(float(r["retrieval_f1"])  for r in srows) / len(srows),
        "generation_em": sum(float(r["generation_em"]) for r in srows) / len(srows),
        "generation_f1": sum(float(r["generation_f1"]) for r in srows) / len(srows),
    }

# Print summary comparison table
header = f"{'System':<20} | {'Retrieval EM':>12} | {'Retrieval F1':>12} | {'Generation EM':>13} | {'Generation F1':>13}"
sep    = "-" * len(header)
print(sep)
print(header)
print(sep)
for sname, metrics in summary.items():
    print(
        f"{sname:<20} | "
        f"{metrics['retrieval_em']:>12.3f} | "
        f"{metrics['retrieval_f1']:>12.3f} | "
        f"{metrics['generation_em']:>13.3f} | "
        f"{metrics['generation_f1']:>13.3f}"
    )
print(sep)
print(f"\nTotal rows evaluated: {len(rows)} ({N_QUESTIONS} questions x {len(summary)} systems)")

## Interpretation

The table above shows whether high retrieval coverage translates to correct LLM-generated answers.

- **High retrieval F1, low generation EM** — the gold answer is present somewhere in the retrieved context, but the LLM fails to extract it correctly. This can happen when the answer is buried deep in a long context, or when the LLM paraphrases rather than quotes.

- **High generation EM, moderate retrieval F1** — the LLM can reason and produce the correct answer even with incomplete context. This suggests the model has some parametric knowledge that supplements the retrieval.

- **HyperRAG vs B1/B2** — if HyperRAG shows higher generation EM/F1, it confirms that 1-hop graph expansion brings in the "bridge" pages needed for multi-hop questions that single-pass FAISS retrieval misses.